# 第十课:MLP / FFN——模型里的知识仓库

讲义:[10-ffn.md](10-ffn.md)。

In [ ]:
import numpy as np

def relu(x):  return np.maximum(x, 0)
def gelu(x):  return 0.5*x*(1+np.tanh(np.sqrt(2/np.pi)*(x+0.044715*x**3)))
def silu(x):  return x/(1+np.exp(-x))

print(f'{"输入":>6} {"ReLU":>8} {"GeLU":>8} {"SiLU":>8}')
for v in (1.0, 2.0, -1.0):
    print(f'{v:6.1f} {relu(v):8.4f} {gelu(v):8.4f} {silu(v):8.4f}')

xs = np.linspace(-3, 3, 5)
print('GeLU 曲线采样:', np.round(gelu(xs), 4))

## 两代 FFN:两层版 vs SwiGLU(讲义 §5)

In [ ]:
rng = np.random.default_rng(3)
d, h = 8, 32            # d=8, 隐藏 32(4 倍演示)
W1 = rng.normal(size=(d, h)) * 0.2
W2 = rng.normal(size=(h, d)) * 0.2
U1 = rng.normal(size=(d, h)) * 0.2      # 门控分支
U2 = rng.normal(size=(h, d)) * 0.2      # 降维:注意是 [h, d]!
U3 = rng.normal(size=(d, h)) * 0.2      # 内容分支

def ffn(x):              return gelu(x @ W1) @ W2
def ffn_swiglu(x):       return (silu(x @ U1) * (x @ U3)) @ U2

x = rng.normal(size=(4, d))          # 4 个词
print('两层版  出入 shape:', x.shape, '->', ffn(x).shape)
print('SwiGLU  出入 shape:', x.shape, '->', ffn_swiglu(x).shape)

# 逐位置独立: 改词1 不影响词2 的输出?
x_mod = np.concatenate([x[:1] + 9, x[1:]])
print('逐位置独立(改词1,词2不变?):', np.allclose(ffn(x)[1:], ffn(x_mod)[1:]))

## 参数账(讲义 §4)

In [ ]:
print('Llama-7B  每层 FFN:', f'{3*4096*11008:,}', f'({3*4096*11008/1e6:.1f}M)')
print('Llama-7B  每层注意力:', f'{4*4096**2:,}', f'({4*4096**2/1e6:.1f}M)')
print('FFN/注意力 参数比:', round(3*4096*11008/(4*4096**2), 2))
print('Llama-70B 每层 FFN:', f'{3*8192*28672:,}', f'({3*8192*28672/1e6:.1f}M)')

## 作业现场

- C2:形状检查器:d、h 给定时打印 SwiGLU 三矩阵形状。
- C3:接进第九课 make_block 堆 4 层。
- D1:80 层全模型 FFN 显存。

In [ ]:
# TODO(C2): 形状检查
def swiglu_shapes(d, hidden):
    # 返回 (W1, W2, W3) 的形状元组
    ...

# print(swiglu_shapes(4096, 11008))

# TODO(D1): 80 层 FFN 参数与显存(bf16)
# total = 80 * 3*8192*28672
# print(total, total*2/1024**3, 'GB')

# TODO(C3): 与 09 课 make_block 组装(把 ffn_swiglu 传入),堆 4 层验证 shape